In [1]:
import pandas as pd 
import numpy as np 
from scipy.stats import stats as st 
import plotly.graph_objects as go 
import plotly.express as px 

In [2]:
df = pd.read_excel(r"C:\works\learnings\AB_testing_framework\online_retail_II.xlsx")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [3]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
df["Customer ID"].nunique()

4383

In [5]:
df["Invoice"].isna().sum()

0

In [6]:
df[df["Quantity"]< 0 ]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
...,...,...,...,...,...,...,...,...
525231,538159,21324,NaN,-18,2010-12-09 17:17:00,0.00,NaN,United Kingdom
525232,538158,20892,NaN,-32,2010-12-09 17:17:00,0.00,NaN,United Kingdom
525234,538161,46000S,Dotcom sales,-100,2010-12-09 17:25:00,0.00,NaN,United Kingdom
525235,538162,46000M,Dotcom sales,-100,2010-12-09 17:25:00,0.00,NaN,United Kingdom


In [7]:
df["Flag"] = np.where(
    df.groupby("Customer ID")["Quantity"].transform("sum") > 0,
    "Yes",
    "No"
)
df.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Flag
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Yes
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Yes
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Yes
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Yes
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Yes
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,Yes
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Yes
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Yes
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,Yes
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,Yes


In [8]:
df["Flag"].value_counts()

Flag
Yes    416803
No     108658
Name: count, dtype: int64

In [9]:
print(min(df["InvoiceDate"]), max(df["InvoiceDate"]))

2009-12-01 07:45:00 2010-12-09 20:01:00


In [10]:
import datetime as dt

In [11]:
months = df["InvoiceDate"].iloc[234].month
months

12

In [12]:
def find_month (date):
    month = date.month
    if month == 12 :
        return "control"
    elif month == 1 :
        return "treatment"
    else :
        return None

In [13]:
find_month(df["InvoiceDate"].iloc[234])

'control'

In [14]:
df["Group"] = df["InvoiceDate"].apply(find_month)
df.head(15)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Flag,Group
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Yes,control
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Yes,control
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Yes,control
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Yes,control
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Yes,control
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,Yes,control
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Yes,control
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Yes,control
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,Yes,control
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,Yes,control


In [15]:
in_both_group = (
    df[df["Group"].isin(["control", "treatment"])]
    .groupby("Customer ID")["Group"]
    .nunique()
)

in_both_group = in_both_group[in_both_group == 2].index

print(len(in_both_group))

429


In [27]:
new_df = df[~df["Customer ID"].isin(in_both_group)]
new_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Flag,Group
31,489437,22143,CHRISTMAS CRAFT HEART DECORATIONS,6,2009-12-01 09:08:00,2.10,15362.0,United Kingdom,Yes,control
32,489437,22145,CHRISTMAS CRAFT HEART STOCKING,6,2009-12-01 09:08:00,2.10,15362.0,United Kingdom,Yes,control
33,489437,22130,PARTY CONE CHRISTMAS DECORATION,12,2009-12-01 09:08:00,0.85,15362.0,United Kingdom,Yes,control
34,489437,21364,PEACE SMALL WOOD LETTERS,2,2009-12-01 09:08:00,6.75,15362.0,United Kingdom,Yes,control
35,489437,21360,JOY LARGE WOOD LETTERS,1,2009-12-01 09:08:00,9.95,15362.0,United Kingdom,Yes,control


In [28]:
new_df["Customer ID"].nunique()

3954

In [29]:
new_df["converted"] = np.where(new_df.groupby("Customer ID")["Quantity"].transform("sum") > 0, 1, 0)
new_df.head()

C:\Users\RITHIK KUMAR\AppData\Local\Temp\ipykernel_20668\2071618797.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["converted"] = np.where(new_df.groupby("Customer ID")["Quantity"].transform("sum") > 0, 1, 0)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Flag,Group,converted
31,489437,22143,CHRISTMAS CRAFT HEART DECORATIONS,6,2009-12-01 09:08:00,2.10,15362.0,United Kingdom,Yes,control,1
32,489437,22145,CHRISTMAS CRAFT HEART STOCKING,6,2009-12-01 09:08:00,2.10,15362.0,United Kingdom,Yes,control,1
33,489437,22130,PARTY CONE CHRISTMAS DECORATION,12,2009-12-01 09:08:00,0.85,15362.0,United Kingdom,Yes,control,1
34,489437,21364,PEACE SMALL WOOD LETTERS,2,2009-12-01 09:08:00,6.75,15362.0,United Kingdom,Yes,control,1
35,489437,21360,JOY LARGE WOOD LETTERS,1,2009-12-01 09:08:00,9.95,15362.0,United Kingdom,Yes,control,1


In [30]:
new_df["converted"].value_counts()

converted
1    281838
0    108656
Name: count, dtype: int64

In [31]:
mean_value = new_df.groupby("converted").agg({
    "Customer ID" : ["mean", "sum"]
}).reset_index()

mean_value.columns = ["converted" , "cus_mean", "cus_sum"]
mean_value.head()

,converted,cus_mean,cus_sum
0,0,14889.204390,1.085423e+07
1,1,15393.170843,4.338380e+09


In [39]:
customer_df = new_df.groupby("Customer ID").agg({
    "Group" : ["first"],
    "converted" : ["max"]
}).reset_index()

customer_df.columns = ["Customer ID","Group_first", "Converted_max"]
customer_df.head()

,Customer ID,Group_first,Converted_max
0,12347.0,control,1
1,12348.0,None,1
2,12349.0,control,1
3,12351.0,None,1
4,12352.0,None,1


In [40]:
customer_df = customer_df.dropna()
customer_df.head()

,Customer ID,Group_first,Converted_max
0,12347.0,control,1
2,12349.0,control,1
9,12358.0,control,1
10,12359.0,control,1
12,12361.0,treatment,1


In [41]:
customer_df["Group_first"].value_counts()

Group_first
control      978
treatment    357
Name: count, dtype: int64

In [42]:
mean_group = customer_df.groupby("Group_first")["Converted_max"].mean()
mean_group

Group_first
control      0.976483
treatment    0.960784
Name: Converted_max, dtype: float64

In [47]:
from statsmodels.stats.proportion import proportions_ztest

control = customer_df[customer_df["Group_first"] == "control"]
treatment = customer_df[customer_df["Group_first"] == "treatment"]

x1 = control["Converted_max"].sum()
n1 = control.shape[0]
x2 = treatment["Converted_max"].sum()
n2 = treatment.shape[0]

stat , p_value = proportions_ztest(
    count=[x1,x2],
    nobs=[n1,n2],
    alternative="two-sided"
)
stat , p_value

(1.5465312186738491, 0.12197632650804853)

In [48]:
if p_value < 0.05 :
    print("There is enough evidence to reject the H0")
else :
    print("There is no enough evidence to reject the H0")

There is no enough evidence to reject the H0
